## Creating a Single Table from Multiple Snapshots

It is common to receive data as **periodic snapshots** — e.g. a termly extract of pupil records, a monthly export from a finance system, or a daily feed from an API. Each snapshot has the same structure but covers a different point in time.

To analyse trends or build a complete picture, you need to combine these into a single table.

In this notebook we’ll initially work with two **synthetic** school snapshots from `catalog_40_copper_analyst_training.bronze`:
* `schools_autumn_2024` — 20 rows, columns include `school_urn`, `school_name`, `city`
* `schools_spring_2025` — 18 rows, but with **different column names** (`urn`, `establishment_name`, `local_authority`)

This is a realistic scenario: the same data provider changed their column naming between extracts.

We'll then progress to automating ingestion of any number of snapshots into a single table for further cleaning. We'll do this first to create a `schools_combined` dataset, breaking down each step of the process, followed by repeating the step for a `pupils_combined` dataset which will become the starting point for cleansing in the `silver` layer.

> **Note:** All data in this module is **entirely synthetic** and does not represent any real schools, pupils, or individuals. No real data has been used.

### Key principles

* **Use `UNION ALL`**, not `UNION`. `UNION` silently removes duplicates, which hides data quality issues and is slower. Always prefer `UNION ALL` and handle duplicates explicitly.
* **Add a snapshot identifier.** Tag each extract so you always know which source a row came from. This is essential for debugging and for time-series analysis.
* **Validate column alignment.** Snapshots may drift over time — columns renamed, added, or removed. Check schemas match before combining.

### Watch out for

* **Overlapping snapshots** — if two extracts cover the same period, you’ll get genuine duplicates
* **Schema drift** — a column called `city` in one snapshot and `local_authority` in another
* **Changed grain** — one snapshot at pupil level, another at pupil-school level

In [0]:
-- Preview both snapshots side by side to spot schema differences
SELECT 'autumn' as snapshot, * FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2024 LIMIT 5;

In [0]:
SELECT 'spring' as snapshot, * FROM catalog_40_copper_analyst_training.bronze.schools_spring_2025 LIMIT 5;

In [0]:
-- The column names differ between snapshots, so a naive UNION ALL would fail or misalign.
-- We must manually map columns to a common schema and tag the source snapshot.

CREATE OR REPLACE TEMP VIEW all_schools AS

SELECT
  school_urn
  ,school_name
  ,city as location
  ,school_type
  ,status
  ,phase
  ,metadata_json
  ,'Autumn 2024' as snapshot_term
FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2024

UNION ALL

SELECT
  urn as school_urn
  ,establishment_name as school_name
  ,local_authority as location
  ,establishment_type as school_type
  ,status
  ,phase_of_education as phase
  ,metadata_json
  ,'Spring 2025' as snapshot_term
FROM catalog_40_copper_analyst_training.bronze.schools_spring_2025;

SELECT * FROM all_schools
ORDER BY school_urn, snapshot_term;

## Combining Snapshots Dynamically

The manual approach above works for two tables, but what if you have **12 monthly extracts**? Or **20 quarterly snapshots**? Manually mapping columns for each one is tedious, error-prone, and breaks the moment a new snapshot arrives with a different schema.

A better approach is to let the database tell you what tables and columns exist, and build the `UNION ALL` query **automatically**. We'll use `INFORMATION_SCHEMA` — a set of system views that describe every table, column, and schema in your catalog.

### The plan

1. **Discover the tables** — find all tables matching a naming pattern (e.g. `schools_*`)
2. **Discover the columns** — get the complete set of column names across all those tables
3. **Check coverage** — see which columns exist in which tables
4. **Build the query** — for each table, SELECT every column from the superset — using the real column where it exists, and `NULL` where it doesn't
5. **Execute** — run the dynamically generated SQL

> **Fiddle:** This section uses `INFORMATION_SCHEMA`, `DECLARE`/`SET VAR`, `EXECUTE IMMEDIATE`, and array functions like `collect_set()`, `array_sort()`, `transform()`, and `array_join()`. These are powerful techniques worth experimenting with — try changing the table pattern or schema to see how the approach adapts.

In [0]:
-- Step 1: Discover all tables matching our pattern
-- INFORMATION_SCHEMA.TABLES lists every table in the catalog
-- The LIKE pattern 'schools_%' will automatically pick up any new snapshots added later

SELECT table_name
FROM catalog_40_copper_analyst_training.information_schema.tables
WHERE table_schema = 'bronze'
  AND table_name LIKE 'schools_%'
ORDER BY table_name;

The query above found all tables in the `bronze` schema whose name starts with `schools_`. If a new snapshot is added next term (e.g. `schools_spring_2026`), this query will automatically pick it up — no code changes required.

### Step 2: Discover the superset of columns

Next, we need to know **every column name that appears in any of those tables**. Some columns will exist in all snapshots, others only in one or two.

In [0]:
-- Step 2: Get every distinct column name across all matching tables
-- This is the SUPERSET of columns that our combined table will need

SELECT DISTINCT column_name, data_type
FROM catalog_40_copper_analyst_training.information_schema.columns
WHERE table_schema = 'bronze'
  AND table_name LIKE 'schools_%'
ORDER BY column_name;

### Step 3: Check which columns exist in which tables

Before building the query, it helps to see the full picture — which columns appear in which tables. This makes the NULL-padding logic concrete: any `-` in the result below will become a `NULL` column in our final query.

We use a `CROSS JOIN` between all distinct columns and all distinct tables to generate every possible combination, then `LEFT JOIN` back to the actual column metadata to check which ones really exist.

In [0]:
-- Step 3: Column presence matrix
-- For each column × table combination, show whether the column exists

SELECT
  ac.column_name,
  t.table_name,
  CASE WHEN c.column_name IS NOT NULL THEN '✓' ELSE '-' END AS present
FROM (
  SELECT DISTINCT column_name
  FROM catalog_40_copper_analyst_training.information_schema.columns
  WHERE table_schema = 'bronze' AND table_name LIKE 'schools_%'
) ac
CROSS JOIN (
  SELECT DISTINCT table_name
  FROM catalog_40_copper_analyst_training.information_schema.tables
  WHERE table_schema = 'bronze' AND table_name LIKE 'schools_%'
) t
LEFT JOIN catalog_40_copper_analyst_training.information_schema.columns c
  ON ac.column_name = c.column_name
  AND t.table_name = c.table_name
  AND c.table_schema = 'bronze'
ORDER BY ac.column_name, t.table_name;

### Step 4: Build the dynamic query

Now for the main event. We need to generate a SQL statement that, for each table, SELECTs every column from the superset — using the real column where it exists, and `NULL AS column_name` where it doesn’t — then joins them all with `UNION ALL`.

This requires several SQL features working together. Rather than throwing them all at you at once, we’ll learn each one individually before combining them at the end.

| Function | Purpose |
| --- | --- |
| `collect_set()` | Collects distinct values into an array |
| `array_sort()` | Sorts the array so columns are in a consistent order |
| `transform()` | Applies a function to each element of an array — like a loop |
| `array_contains()` | Checks if a value exists in an array |
| `array_join()` | Joins array elements into a string with a separator |
| `DECLARE` / `SET VAR` | Creates a SQL variable to hold a value for reuse |
| `EXECUTE IMMEDIATE` | Runs a SQL string as a live query |

We’ll save intermediate results into **session variables** as we go. This means each step builds on the previous one without repeating code.

---

#### 4a: `collect_set()` and `array_sort()` — collecting values into arrays

`collect_set()` is an aggregate function that gathers all **distinct** values from a column into a single **array**. It’s similar to `DISTINCT`, but instead of returning separate rows, it packs everything into one value.

`array_sort()` then sorts that array alphabetically. This is important because we need columns in the **same order** for every table’s SELECT statement — otherwise the `UNION ALL` would misalign columns.

We’ll save the sorted superset into a variable called `all_cols` so we can reuse it in subsequent steps.

In [0]:
-- 4a: Collect all distinct column names into a sorted array and save it

DECLARE OR REPLACE all_cols ARRAY<STRING>;

SET VAR all_cols = (
  SELECT array_sort(collect_set(column_name))
  FROM catalog_40_copper_analyst_training.information_schema.columns
  WHERE table_schema = 'bronze' AND table_name LIKE 'schools_%'
);

SELECT all_cols AS sorted_column_superset;

#### 4b: `collect_set()` per table — saving a single table’s columns

We’ve already saved the full column superset in `all_cols`. Now we need to know which of those columns a **specific table** actually has.

We’ll use `collect_set()` again — this time filtered to a single table — and save the result into a variable called `table_cols`.

In [0]:
-- 4b: Save one table's column set into a variable

DECLARE OR REPLACE table_cols ARRAY<STRING>;

SET VAR table_cols = (
  SELECT collect_set(column_name)
  FROM catalog_40_copper_analyst_training.information_schema.columns
  WHERE table_schema = 'bronze' AND table_name = 'schools_autumn_2024'
);

SELECT table_cols AS autumn_2024_columns;

#### 4c: `transform()` and `array_contains()` — iterating over arrays with conditions

`transform()` takes an array and applies a function to **every element**, returning a new array of the same length. Think of it as a `FOR EACH` loop. The syntax is:

```sql
transform(array, element -> expression)
```

The `element ->` part is called a **lambda** — a small inline function. The name before the arrow (here `element`) is a placeholder that takes the value of each array item in turn.

`array_contains()` checks whether a specific value exists in an array, returning `TRUE` or `FALSE`.

Now that we have both `all_cols` (the full superset) and `table_cols` (one table’s columns), we can use `transform()` to iterate over the superset and check which columns `schools_autumn_2024` actually has. We’ll save the result into `column_check`.

In [0]:
-- 4c: Check which superset columns exist in this table

DECLARE OR REPLACE column_check ARRAY<STRING>;

SET VAR column_check = (
  SELECT transform(
    all_cols,
    col -> concat(
      col, ' → ',
      CASE WHEN array_contains(table_cols, col) THEN '✓ exists' ELSE '✗ missing' END
    )
  )
);

SELECT column_check;

#### 4d: `array_join()` and `concat()` — building a SQL statement from an array

`array_join()` takes an array and concatenates all elements into a single string, with a separator between each element:

```sql
array_join(array, separator) → 'element1, element2, element3'
```

This is the bridge between arrays and usable SQL. We swap the human-readable `✓ exists / ✗ missing` labels from 4c for actual SQL expressions — either `` `city` `` (for columns that exist) or `` NULL AS `establishment_name` `` (for columns that don't exist in that table) — then use `array_join()` with `', '` as the separator to assemble them into a complete `SELECT` clause.

Because `all_cols` and `table_cols` are already saved as variables, we just need to wrap the result in `concat()` to produce a full SQL statement. We’ll save it into `select_clause`.

In [0]:
-- 4d: Build a complete SELECT statement for one table

DECLARE OR REPLACE select_clause STRING;

SET VAR select_clause = (
  SELECT concat(
    'SELECT ',
    array_join(
      transform(
        all_cols,
        col -> CASE
          WHEN array_contains(table_cols, col) THEN concat('`', col, '`')
          ELSE concat('NULL AS `', col, '`')
        END
      ),
      ', '
    ),
    ' FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2024'
  )
);

SELECT select_clause AS generated_select;

#### 4e: Scaling up — building a SELECT for every table

Steps 4a–4d demonstrated each building block on a **single table**. Now we apply the same `transform()` + `array_join()` pattern to **every** matching table at once, collecting the results into an array of complete SELECT statements.

We reuse `all_cols` (from 4a) directly. The only new work is a CTE that gathers each table’s column set (the multi-table equivalent of `table_cols` from 4b).

We also introduce `q` — a variable holding a single-quote character via `char(39)`. This avoids the fiddly `''''` escaping that comes from embedding quotes inside quoted strings.

In [0]:
-- 4e: Build a SELECT statement for each matching table

DECLARE OR REPLACE q STRING DEFAULT char(39);
DECLARE OR REPLACE per_table_selects ARRAY<STRING>;

SET VAR per_table_selects = (
  WITH table_cols AS (
    SELECT table_name, collect_set(column_name) AS existing_cols
    FROM catalog_40_copper_analyst_training.information_schema.columns
    WHERE table_schema = 'bronze' AND table_name LIKE 'schools_%'
    GROUP BY table_name
  )
  SELECT collect_list(
    concat(
      'SELECT ',
      array_join(
        transform(
          all_cols,
          col -> CASE
            WHEN array_contains(existing_cols, col)
              THEN concat('`', col, '`')
            ELSE concat('NULL AS `', col, '`')
          END
        ),
        ', '
      ),
      ', ', q, table_name, q, ' AS source_table',
      ' FROM catalog_40_copper_analyst_training.bronze.', table_name
    )
  )
  FROM table_cols
);

SELECT per_table_selects;

#### 4f: Combining with `UNION ALL`

The final step is simple: join the array of SELECT statements into a single string separated by `UNION ALL`.

In [0]:
-- 4f: Join all per-table SELECTs into a single query

DECLARE OR REPLACE dynamic_sql STRING;

SET VAR dynamic_sql = (
  SELECT array_join(per_table_selects, '\nUNION ALL\n')
);

SELECT dynamic_sql AS generated_query;

### Step 5: Execute the generated query

The variable `dynamic_sql` now holds a complete, valid SQL query. `EXECUTE IMMEDIATE` runs it as if you had typed it directly into a cell.

> **Tip:** Because we used `DECLARE OR REPLACE`, you can safely re-run the cells above without restarting the session. The variable will simply be overwritten.

In [0]:
-- Step 5: Run the dynamically generated query
EXECUTE IMMEDIATE dynamic_sql;

### Step 6: Store the result as a reusable view

Step 5 executed the query and displayed the results, but they aren’t stored anywhere we can query later. By prepending `CREATE OR REPLACE TEMPORARY VIEW schools_combined AS` to our existing `dynamic_sql` string, we can persist the combined dataset for the rest of the session.

In [0]:
-- Step 6: Prepend CREATE VIEW to the dynamic query and execute it

DECLARE OR REPLACE create_view_sql STRING;

SET VAR create_view_sql = concat(
  'CREATE OR REPLACE TEMPORARY VIEW schools_combined AS\n',
  dynamic_sql
);

EXECUTE IMMEDIATE create_view_sql;

In [0]:
-- Verify the view is queryable
SELECT * FROM schools_combined;

### What just happened?

With no hardcoded column names or table names, we:

1. Found all `schools_*` tables in the `bronze` schema automatically
2. Built the complete column superset across all of them
3. Generated a `UNION ALL` that pads missing columns with `NULL`
4. Executed the result

If a new snapshot appears next term — with yet another set of column names — **this code handles it without modification**. That's the power of dynamic SQL.

> **Next steps:** The combined table still has all the messiness from the individual snapshots — duplicate rows, inconsistent labels, and different column names for the same concept (e.g. `school_name` vs `establishment_name`). We are not going to worry about reconciling renamed columns here — the following notebooks will address cleaning issues systematically. The important thing is that **every column and every row from every snapshot is captured in one place**.

### Step 7: Write to Silver as a permanent table

The temporary view from Step 6 disappears when the session ends. To persist our combined dataset as the **first step of the cleaning pipeline**, we write it to the `silver` schema as a proper Delta table.

This follows the medallion architecture: raw snapshots live in `bronze`, and the combined (but still messy) dataset moves to `silver` where subsequent notebooks will clean it.

> **Note:** The cell below creates the SQL statement needed to store the table in the `silver` schema of `catalog_40_copper_analyst_training`. The catalog is read-only though, so trying to execute that statement in this notebook would fail. The permanent table has already been created in the `silver` schema though so that we can pick it up in the following notebooks.

In [0]:
-- Step 7: Persist the combined dataset as a Silver table

DECLARE OR REPLACE create_table_sql STRING;

SET VAR create_table_sql = concat(
  'CREATE OR REPLACE TABLE catalog_40_copper_analyst_training.silver.schools_combined AS\n',
  dynamic_sql
);

SELECT create_table_sql;

In [0]:
-- Verify the table exists and is queryable
SELECT source_table, COUNT(*) AS row_count
FROM catalog_40_copper_analyst_training.silver.schools_combined
GROUP BY source_table
ORDER BY source_table;


## Creating a single pupils table

We've collated all of the schools data into a single table in silver now. The schools table gives us information about schools, but to get data about the students in those schools we'll need to collate all the pupil data into a single table now too.

The cell below does this in a single step. `EXECUTE IMMEDIATE` requires a variable, so we can't avoid `DECLARE`/`SET` entirely, but we can compress the entire pipeline into three dense lines with no intermediate steps or displays.

Notice how **much harder it is to read and understand** compared to the step-by-step approach above. The logic is identical — the only difference is presentation. This is why it’s **not recommended** to take this approach. Breaking code into component parts with named variables and intermediate outputs makes it far easier to debug, maintain, and hand over to colleagues.

> **Note:** As above the cell below creates the SQL statement needed to store the table in the `silver` schema of `catalog_40_copper_analyst_training` but does not execute it. The table already exists in the `silver` schema to be picked up in later notebooks though.

In [0]:
-- Combine all pupils_* tables into a single silver table in one go

DECLARE OR REPLACE _sql STRING;

SET VAR _sql = (
  WITH
  _all_cols AS (
    SELECT array_sort(collect_set(column_name)) AS cols
    FROM catalog_40_copper_analyst_training.information_schema.columns
    WHERE table_schema = 'bronze' AND table_name LIKE 'pupils_%'
  ),
  _table_cols AS (
    SELECT table_name, collect_set(column_name) AS existing_cols
    FROM catalog_40_copper_analyst_training.information_schema.columns
    WHERE table_schema = 'bronze' AND table_name LIKE 'pupils_%'
    GROUP BY table_name
  ),
  _per_table AS (
    SELECT concat(
      'SELECT ',
      array_join(
        transform(
          ac.cols,
          col -> CASE
            WHEN array_contains(tc.existing_cols, col) THEN concat('`', col, '`')
            ELSE concat('NULL AS `', col, '`')
          END
        ),
        ', '
      ),
      ', ', char(39), tc.table_name, char(39), ' AS source_table',
      ' FROM catalog_40_copper_analyst_training.bronze.', tc.table_name
    ) AS stmt
    FROM _table_cols tc
    CROSS JOIN _all_cols ac
  )
  SELECT concat(
    'CREATE OR REPLACE TABLE catalog_40_copper_analyst_training.silver.pupils_combined AS\n',
    array_join(collect_list(stmt), '\nUNION ALL\n')
  )
  FROM _per_table
);

SELECT _sql;

In [0]:
-- Verify the pupils table was created
SELECT source_table, COUNT(*) AS row_count
FROM catalog_40_copper_analyst_training.silver.pupils_combined
GROUP BY source_table
ORDER BY source_table;